# openbatrangs on Google Colab

In this notebook, we install openbatrangs, run a coding model with Ollama on
Colab's GPU, and give the agent a task in a repository.

Before you start, choose **Runtime → Change runtime type → T4 GPU**. Without a
GPU, the model runs on the CPU and each step takes minutes.

## Install openbatrangs

The installer downloads the release binary for Linux and checks its SHA-256
checksum. As root, which you are on Colab, it installs to `/usr/local/bin`.

In [ ]:
!curl -fsSL https://github.com/arpanpathak/openbatrangs/releases/latest/download/install.sh | sh
!openbatrangs --version

## Start Ollama

We install Ollama and start `ollama serve` in its own session, so the server
keeps running after this cell ends. The installer needs `zstd` to extract
Ollama and `lspci` to find the GPU. The server log goes to
`/content/ollama.log`.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request

ollama_log = open("/content/ollama.log", "w")
subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log, start_new_session=True)

for attempt in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version")
        print("Ollama is running.")
        break
    except OSError:
        time.sleep(1)
else:
    print("Ollama did not start; see /content/ollama.log.")

## Download a model

A T4 has 15 GB of memory. `qwen2.5-coder:7b` takes about 5 GB, so it leaves room
for a long context. Any Ollama model of up to about 12 GB fits, for example
`qwen2.5-coder:14b`.

In [ ]:
MODEL = "qwen2.5-coder:7b"  # @param {type:"string"}

!ollama pull {MODEL}

## Get a repository

We clone the repository the agent works in to `/content/project`. To work on
your own code, change `REPO`, or upload files to `/content/project` with the
file browser on the left.

In [ ]:
REPO = "https://github.com/arpanpathak/openbatrangs"  # @param {type:"string"}

!rm -rf /content/project && git clone --depth 1 {REPO} /content/project

## Run one task

A notebook cell has no terminal, so we pass the task on the command line. The
agent prints its steps and its answer here. Without `--confirm`, the agent
writes files and runs commands without asking, so give it a copy of your code.

In [ ]:
TASK = "Explain how this project is organized, in a short list"  # @param {type:"string"}

!openbatrangs --no-update-check --cwd /content/project --model {MODEL} "{TASK}"

## Open the terminal UI

The terminal UI needs a real terminal. `colab-xterm` opens one inside the next
cell. In it, run the command below.

```sh
openbatrangs --cwd /content/project --model qwen2.5-coder:7b
```

In [ ]:
!pip -q install colab-xterm
%load_ext colabxterm
%xterm

## Use another model server

If you already serve a model with vLLM, SGLang, llama.cpp or TensorRT-LLM, you
can skip Ollama. Point openbatrangs at the server's OpenAI-compatible URL
instead. Store the key with the key icon on the left, as a secret named
`MODEL_API_KEY`.

```python
from google.colab import userdata
open("/content/api-key", "w").write(userdata.get("MODEL_API_KEY"))
```

```sh
!openbatrangs --openai-url https://your-server/v1 --api-key-file /content/api-key "your task"
```